# Chapter 7: Web Scraping with requests and Beautiful Soup

Companion code for *Python for Data Science* by Vamsi Krishna Kandala. Run the cells in order.

## 7.1 From page to table

In [ ]:
import practice_web
import requests
from bs4 import BeautifulSoup

BASE = practice_web.start()
response = requests.get(f"{BASE}/shop/", timeout=10)
soup = BeautifulSoup(response.text, "lxml")
print(response.status_code, soup.title.get_text())

## 7.2 Looking at a page like a scraper

In [ ]:
card = soup.select_one("article.book")
print(card.prettify()[:620])

## 7.3 CSS selectors in depth

In [ ]:
print(len(soup.select("article.book")))
print(len(soup.select("p.stock.out-of-stock")))
print(soup.select_one("ul.pager > li.current").get_text())
print([a.get_text() for a in soup.select("ul.categories a")][:4])

## 7.4 Text, attributes and links

In [ ]:
link = card.select_one("h3 a")
print(link["href"])
print(card.select_one("p.stars")["class"])
for a in soup.select("article.book h3 a")[1:4]:
    print(a.get_text(strip=True), "|", a["title"])

In [ ]:
from pprint import pprint
from urllib.parse import urljoin

page_url = f"{BASE}/shop/catalogue/page-1.html"
print(urljoin(page_url, link["href"]))
print(urljoin(page_url, "page-2.html"))

In [ ]:
def parse_card(card, page_url):
    """Return the raw values of one book card as a dictionary."""
    link = card.select_one("h3 a")
    classes = card.select_one("p.stars")["class"]
    text = lambda css: card.select_one(css).get_text(strip=True)
    return {"title": link["title"],
            "url": urljoin(page_url, link["href"]),
            "author": text("p.author"),
            "rating": [c for c in classes if "stars-" in c][0],
            "price": text("p.price"),
            "stock": text("p.stock")}

raw = [parse_card(c, page_url) for c in soup.select("article.book")]
pprint(raw[0], width=68)

## 7.5 Cleaning scraped values

In [ ]:
import pandas as pd

books = pd.DataFrame(raw)
print(books.dtypes.value_counts())
print(books[["rating", "price", "stock"]].head(3))

In [ ]:
clean = books.assign(
    author=books["author"].str.removeprefix("by "),
    rating=books["rating"].str.removeprefix("stars-").astype(int),
    price=pd.to_numeric(books["price"].str.removeprefix("$")),
    in_stock=books["stock"].eq("In stock"),
).drop(columns="stock")
print(clean.dtypes)

In [ ]:
print(clean[["title", "rating", "price", "in_stock"]].head(3))
print(clean.groupby("rating")["price"].mean().round(2))

## 7.6 Tables with read_html

In [ ]:
from io import StringIO

page = requests.get(f"{BASE}/data/life-expectancy-2007.html",
                    timeout=10).text
tables = pd.read_html(StringIO(page), thousands=",")
print(len(tables))
print(tables[0].head(3))
print(tables[0].dtypes)

In [ ]:
page_soup = BeautifulSoup(page, "lxml")
continents = [h2.get_text() for h2 in page_soup.select("h2")]
world = pd.concat(
    [t.assign(continent=c) for c, t in zip(continents, tables)],
    ignore_index=True)
world["GDP per person"] = pd.to_numeric(
    world["GDP per person"].str.replace("[$,]", "", regex=True))
print(world.shape)
print(world.groupby("continent")["Life expectancy"].median())

## 7.7 Putting it together: a catalog page to a clean table

In [ ]:
def scrape_listing(url, session=requests):
    """Return the raw book records on one catalog or category page."""
    response = session.get(url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "lxml")
    return [parse_card(c, url) for c in soup.select("article.book")]

def clean_books(raw):
    """Turn raw book records into a typed DataFrame."""
    books = pd.DataFrame(raw)
    return books.assign(
        author=books["author"].str.removeprefix("by "),
        rating=books["rating"].str.removeprefix("stars-").astype(int),
        price=pd.to_numeric(books["price"].str.removeprefix("$")),
        in_stock=books["stock"].eq("In stock"),
    ).drop(columns="stock")

In [ ]:
url = f"{BASE}/shop/category/poetry/page-1.html"
poetry = clean_books(scrape_listing(url))
print(len(poetry), poetry["price"].mean().round(2))
cols = ["title", "rating", "price"]
print(poetry.nlargest(3, "rating")[cols].to_string(index=False))

## 7.8 Key terms at a glance